In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import *

apply_plot_style()
FIGURES_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = load_raw()
print(f"{len(df)} records, {len(COLUMNS)} columns")
df[SITE_COL].value_counts()

In [ ]:
df[TARGET] = (df["num"] > 0).astype(int)
feats = feature_columns(df)
print(f"{len(feats)} features")
print(f"class balance: {int((df[TARGET]==0).sum())} negative / "
      f"{int((df[TARGET]==1).sum())} positive "
      f"({df[TARGET].mean()*100:.1f}% positive)")

In [ ]:
audit = pd.DataFrame({
    "missing": df[feats].isna().sum(),
    "missing_pct": (df[feats].isna().mean() * 100).round(1),
    "distinct": df[feats].nunique(),
}).sort_values("missing_pct", ascending=False)

complete = int(df[feats].notna().all(axis=1).sum())
print(f"complete cases: {complete} of {len(df)} ({complete/len(df)*100:.1f}%)")
audit

In [ ]:
audit = build_audit(df)
audit_table_figure(df, audit, FIGURES_DIR / "table_2_1_data_quality_audit.png")
audit.to_csv(FIGURES_DIR / "table_2_1_data_quality_audit.csv", index=False)

audit

In [ ]:
pct = df.groupby(SITE_COL)[feats].apply(lambda g: g.isna().mean() * 100)
pct.loc["ALL"] = df[feats].isna().mean() * 100

fig, ax = plt.subplots(figsize=(0.52 * len(feats) + 2.6, 0.5 * len(pct) + 2))
im = ax.imshow(pct, cmap=seq_cmap(), vmin=0, vmax=100, aspect="auto")
for (r, c), v in np.ndenumerate(pct.to_numpy()):
    ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=7.5,
            color="white" if v > 55 else "#0b0b0b")
ax.set_xticks(range(len(feats)), feats, rotation=90, fontsize=8)
ax.set_yticks(range(len(pct)), pct.index, fontsize=8.5)
ax.set_xticks(np.arange(len(feats) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(pct) + 1) - 0.5, minor=True)
ax.grid(which="minor", color="#fcfcfb", linewidth=1.4)
ax.tick_params(which="minor", length=0)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.7, label="% missing")
ax.set_title("Missing values by hospital", loc="left", fontsize=11, pad=12)
fig.savefig(FIGURES_DIR / "eda_missingness_by_site.png")
plt.show()

In [ ]:
order = df[feats].isna().sum().sort_values(ascending=False).index.tolist()
grid = df.sort_values(SITE_COL)[order].isna().astype(int)

from matplotlib.colors import LinearSegmentedColormap
fig, ax = plt.subplots(figsize=(0.5 * len(order) + 2.4, 5))
ax.imshow(grid, cmap=LinearSegmentedColormap.from_list("m", ["#eef4fd", "#184f95"]),
          aspect="auto", interpolation="nearest")
boundary = 0
for site, count in df[SITE_COL].value_counts().reindex(
        sorted(df[SITE_COL].unique())).items():
    boundary += count
    ax.axhline(boundary - 0.5, color="#eb6834", linewidth=1.2)
    ax.text(len(order) - 0.3, boundary - count / 2, site, fontsize=7.5,
            va="center", ha="left", color="#52514e")
ax.set_xticks(range(len(order)), order, rotation=90, fontsize=8)
ax.set_ylabel("Records, grouped by hospital")
ax.set_yticks([])
ax.grid(False)
ax.set_title("Missing-value pattern  (dark = missing)", loc="left", fontsize=11, pad=12)
fig.savefig(FIGURES_DIR / "eda_missingness_pattern.png")
plt.show()

In [ ]:
cols = feats + [TARGET]
corr = df[cols].corr()
n = len(cols)

fig, ax = plt.subplots(figsize=(0.52 * n + 2.2, 0.52 * n + 1.6))
im = ax.imshow(corr, cmap=div_cmap(), vmin=-1, vmax=1)
for (r, c), v in np.ndenumerate(corr.to_numpy()):
    if r != c:
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if abs(v) > 0.55 else "#0b0b0b")
ax.set_xticks(range(n), cols, rotation=90, fontsize=8)
ax.set_yticks(range(n), cols, fontsize=8)
ax.set_xticks(np.arange(n + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(n + 1) - 0.5, minor=True)
ax.grid(which="minor", color="#fcfcfb", linewidth=1.4)
ax.tick_params(which="minor", length=0)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.7, label="Pearson r")
ax.set_title("Feature correlations", loc="left", fontsize=11, pad=12)
fig.savefig(FIGURES_DIR / "eda_correlation.png")
plt.show()

In [ ]:
cols_n = 4
rows_n = int(np.ceil(len(feats) / cols_n))
fig, axes = plt.subplots(rows_n, cols_n, figsize=(3.0 * cols_n, 2.3 * rows_n))
axes = np.atleast_1d(axes).ravel()

for ax, c in zip(axes, feats):
    if df[c].nunique(dropna=True) > CONTINUOUS_MIN_UNIQUE:
        bins = np.linspace(df[c].min(), df[c].max(), 20)
        for k in (0, 1):
            vals = df.loc[df[TARGET] == k, c].dropna()
            ax.hist(vals, bins=bins, density=True, histtype="stepfilled",
                    facecolor=CLASS_COLOURS[k], alpha=0.32)
            ax.hist(vals, bins=bins, density=True, histtype="step",
                    edgecolor=CLASS_COLOURS[k], linewidth=1.6)
        ax.set_yticks([])
    else:
        levels = sorted(df[c].dropna().unique())
        x = np.arange(len(levels))
        for k in (0, 1):
            sub = df.loc[df[TARGET] == k, c]
            share = [(sub == lv).sum() / max(1, sub.notna().sum()) for lv in levels]
            ax.bar(x + (k - 0.5) * 0.38, share, 0.34, color=CLASS_COLOURS[k],
                   edgecolor="white", linewidth=0.8)
        ax.set_xticks(x, [f"{lv:g}" for lv in levels], fontsize=7.5)
        ax.set_ylim(0, 1)
        ax.tick_params(axis="y", labelsize=7)
    ax.set_title(c, fontsize=9.5)
    ax.tick_params(axis="x", labelsize=7.5)
    ax.grid(False)
for ax in axes[len(feats):]:
    ax.axis("off")

handles = [plt.Line2D([], [], color=CLASS_COLOURS[k], linewidth=6, label=lab)
           for k, lab in enumerate(["No heart disease", "Heart disease"])]
fig.legend(handles=handles, loc="lower right", fontsize=9, bbox_to_anchor=(0.98, 0.02))
fig.suptitle("Feature distributions by class", x=0.02, y=1.005, ha="left", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_distributions.png")
plt.show()

In [ ]:
zeros = df[df["chol"] == 0]
print(f"{len(zeros)} records with chol == 0")
print(zeros[SITE_COL].value_counts().to_string())
print()
print("disease rate by hospital:")
print(df.groupby(SITE_COL)[TARGET].agg(["size", "mean"]).round(3).to_string())

In [ ]:
summary = pd.DataFrame([{
    "feature": c,
    "missing": int(df[c].isna().sum()),
    "missing_pct": round(df[c].isna().mean() * 100, 1),
    "distinct": int(df[c].nunique()),
    "mean": round(df[c].mean(), 2),
    "std": round(df[c].std(), 2),
    "min": df[c].min(),
    "median": df[c].median(),
    "max": df[c].max(),
    "corr_with_target": round(df[[c, TARGET]].corr().iloc[0, 1], 3),
} for c in feats]).set_index("feature")

summary.to_csv(FIGURES_DIR / "eda_feature_summary.csv")
summary